# Chatbot con MCP, Ollama y Gradio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion6/3-chatbot-con-mcp-y-gradio.ipynb)

Este notebook es un ejemplo avanzado y opcional de la sesión. No introduce conceptos nuevos: simplemente toma el agente con herramientas MCP del notebook anterior y le monta encima una interfaz conversacional con Gradio, siguiendo el estilo visto antes en la sesión de RAG con LangChain. La idea es cerrar la unidad mostrando cómo pasar de una integración técnica a una experiencia de usuario lista para demostración.

### Referencias
- [Gradio](https://www.gradio.app/)
- [MCP Python SDK](https://py.sdk.modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://pypi.org/project/langchain-mcp-adapters/)
- [Ollama](https://ollama.com/)


In [1]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
# En Colab, cuando se abre el notebook desde GitHub, los archivos auxiliares del repo
# no siempre se descargan. Esta celda garantiza que mcp_servers exista.
is_colab = bool(globals().get('IN_COLAB', False))
if is_colab:
    !mkdir -p Sesion6/mcp_servers
    !wget -q -O Sesion6/mcp_servers/__init__.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/__init__.py
    !wget -q -O Sesion6/mcp_servers/calculator_mcp_server.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/calculator_mcp_server.py
    !wget -q -O Sesion6/mcp_servers/weather_mcp_server.py https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion6/mcp_servers/weather_mcp_server.py
    print('MCP servers descargados para Colab.')
else:
    print('Entorno local detectado: no se descargan archivos auxiliares.')

Entorno local detectado: no se descargan archivos auxiliares.


In [3]:
!test '{IN_COLAB}' = 'True' && pip install "mcp>=1.24.0,<2.0.0" "langchain-mcp-adapters>=0.3.0,<0.4.0" langchain langchain-core langchain-ollama langgraph httpx gradio ollama colab-xterm

# En local, instala/actualiza con: pip install -r requirements.txt

### Cargando a Ollama

Usaremos el mismo modelo local y la misma idea de tools de los notebooks previos para que lo único nuevo aquí sea la interfaz.


In [ ]:
!sudo apt install zstd -y
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi


## Atención

En Colab recuerda arrancar `ollama serve` desde la terminal embebida si todavía no está corriendo. En local basta con tener el servicio levantado de antemano.

In [5]:
#%load_ext colabxterm
#%xterm


Mantendremos `llama3.2:3b` para la demostración.


In [19]:
!ollama pull llama3.2:3b


pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


## Reutilizamos los mismos servidores MCP

Para que el notebook sea transparente y fácil de depurar, **no** ocultamos la implementación en cadenas largas. Reutilizamos los mismos archivos Python del repositorio para calculadora y clima, igual que en el notebook anterior.

Si el notebook se abre en Colab desde GitHub, ejecuta primero la celda de bootstrap para descargar estos archivos auxiliares en la sesión temporal. En local, el flujo actual se mantiene sin cambios.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
candidate_dirs = [
    REPO_ROOT / 'Sesion6' / 'mcp_servers',
    REPO_ROOT / 'mcp_servers',
]

SERVERS_DIR = next((path for path in candidate_dirs if path.exists()), None)
if SERVERS_DIR is None:
    msg = (
        'No se encontró la carpeta mcp_servers. '
        'Si estás en Colab, ejecuta primero la celda de bootstrap que descarga los servidores MCP.'
    )
    raise FileNotFoundError(msg)

calculator_server = SERVERS_DIR / 'calculator_mcp_server.py'
weather_server = SERVERS_DIR / 'weather_mcp_server.py'

if not calculator_server.exists() or not weather_server.exists():
    msg = (
        f'Faltan archivos MCP en {SERVERS_DIR}. '
        'Si estás en Colab, vuelve a ejecutar la celda de bootstrap.'
    )
    raise FileNotFoundError(msg)

SERVERS_DIR

In [ ]:
print(f'Servidor calculadora: {calculator_server}')
print('-' * 80)
print(calculator_server.read_text(encoding='utf-8'))

In [ ]:
print(f'Servidor clima: {weather_server}')
print('-' * 80)
print(weather_server.read_text(encoding='utf-8'))

## Inicio y apagado de servidores MCP

Tienes dos formas válidas de ejecutar los servidores:

1. Modo automático (recomendado aquí): el cliente MCP por `stdio` levanta y cierra procesos durante la llamada.
2. Modo manual: inicia cada servidor en terminal separada y detén con `Ctrl+C` al finalizar.

Si usas modo manual, recuerda apagar procesos para evitar sesiones colgadas.

In [ ]:
import sys

manual_start_commands = [
    f"{sys.executable} {calculator_server}",
    f"{sys.executable} {weather_server}",
]

print('Comandos para modo manual (ejecutar en terminales separadas):')
for cmd in manual_start_commands:
    print(f'- {cmd}')

print('\nPara detener cada servidor manual: Ctrl+C en su terminal.')

## Construimos el agente consumidor de herramientas MCP

Igual que antes, el agente no conoce directamente las funciones de Python; descubre las capacidades a través de los servidores MCP y decide cuándo invocarlas.

Aquí usamos la API soportada `create_agent` de LangChain para evitar dependencias en rutas deprecadas.

### Cómo decide el agente usar tools

En modo autónomo, el modelo puede responder directamente o llamar tools. Que una tool esté disponible **no garantiza** su uso en todos los turnos.

Para mejorar la probabilidad de uso de tool sin forzar:

1. Escribe prompts con intención operativa clara (por ejemplo: "usa la herramienta de clima").
2. Define descripciones de tool precisas y accionables.
3. Usa instrucciones de sistema que prioricen tools cuando haya datos externos.
4. Mantén una ruta guiada/determinística para tareas donde sí necesitas evidencia MCP obligatoria.

In [8]:
import io
import subprocess
import anyio

def apply_global_subprocess_patch():
    is_colab = bool(globals().get('IN_COLAB', False))
    if not is_colab:
        return 'Global Popen patch skipped outside Colab'

    original_popen = subprocess.Popen

    if getattr(original_popen, '_colab_patched', False):
        return 'Global Popen patch already active'

    def _patched_popen(*args, **kwargs):
        # Colab notebook streams can expose fileno() but still fail at runtime.
        for stream_name in ['stdin', 'stdout', 'stderr']:
            stream = kwargs.get(stream_name)
            if stream is None:
                continue
            try:
                if hasattr(stream, 'fileno'):
                    stream.fileno()
            except (io.UnsupportedOperation, AttributeError):
                kwargs[stream_name] = subprocess.PIPE
        return original_popen(*args, **kwargs)

    _patched_popen._colab_patched = True
    subprocess.Popen = _patched_popen
    return 'Global Popen patch applied'

def ensure_colab_anyio_open_process_patch() -> bool:
    is_colab = bool(globals().get('IN_COLAB', False))
    if not is_colab:
        return False

    if getattr(anyio.open_process, '_mcp_colab_safe', False):
        return True

    original_open_process = anyio.open_process

    async def _colab_safe_open_process(*args, **kwargs):
        kwargs.setdefault('stdin', subprocess.PIPE)
        kwargs.setdefault('stdout', subprocess.PIPE)
        kwargs.setdefault('stderr', subprocess.PIPE)
        try:
            return await original_open_process(*args, **kwargs)
        except io.UnsupportedOperation as exc:
            if 'fileno' not in str(exc).lower():
                raise
            return await original_open_process(*args, **kwargs)

    _colab_safe_open_process._mcp_colab_safe = True
    anyio.open_process = _colab_safe_open_process
    return True

print(apply_global_subprocess_patch())
MCP_COLAB_PATCH_APPLIED = ensure_colab_anyio_open_process_patch()
print('Parche Colab fileno activo:', MCP_COLAB_PATCH_APPLIED)

Global Popen patch skipped outside Colab
Parche Colab fileno activo: False


In [17]:
import re
import sys
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_ollama import ChatOllama

if 'apply_global_subprocess_patch' in globals():
    print(apply_global_subprocess_patch())

MODEL = 'llama3.2:3b'
llm = ChatOllama(model=MODEL, temperature=0)

client = MultiServerMCPClient(
    {
        'calculadora': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(calculator_server)],
        },
        'clima': {
            'transport': 'stdio',
            'command': sys.executable,
            'args': [str(weather_server)],
        },
    }
)

async def load_mcp_tools_with_diagnostics(mcp_client):
    is_colab = bool(globals().get('IN_COLAB', False))
    try:
        return await mcp_client.get_tools()
    except Exception as exc:
        if is_colab and 'fileno' in str(exc).lower():
            patched = globals().get('MCP_COLAB_PATCH_APPLIED', False)
            raise RuntimeError(
                'Colab no pudo abrir subprocess stdio (fileno). '
                f'Parche activo={patched}. Ejecuta la celda de compatibilidad Colab y reintenta.'
            ) from exc
        raise

mcp_tools = await load_mcp_tools_with_diagnostics(client)
agent_mcp = create_agent(model=llm, tools=mcp_tools)

def available_tool_names():
    return [tool.name for tool in mcp_tools]

def tool_from_name(tool_name: str):
    return next((tool for tool in mcp_tools if tool.name == tool_name), None)

def detect_suggested_tool(question: str) -> str | None:
    lower = question.lower()
    weather_signals = ['clima', 'temperatura', 'humedad', 'viento', 'weather', 'lluvia']
    if any(word in lower for word in weather_signals):
        return 'clima_actual'
    has_digit = any(ch.isdigit() for ch in question)
    has_operator = any(op in question for op in ['+', '-', '*', '/', '%', '(', ')'])
    if has_digit and has_operator:
        return 'calculadora'
    return None

def select_tool_name(question: str, preferred_tool: str):
    if preferred_tool and preferred_tool != 'Auto':
        return preferred_tool
    return detect_suggested_tool(question)

def infer_calculator_expression(question: str) -> str | None:
    # Extrae una expresion aritmetica probable desde el texto del usuario.
    candidates = re.findall(r"[0-9\s\+\-\*\/%\(\)\.]+", question)
    candidates = [c.strip() for c in candidates if any(ch.isdigit() for ch in c) and any(op in c for op in '+-*/%')]
    if not candidates:
        return None
    return max(candidates, key=len)

def infer_city(question: str) -> str:
    # Heuristica simple: toma texto tras 'en ...' y limpia signos frecuentes.
    match = re.search(r"\ben\s+([^\?\.!\n]+)", question, flags=re.IGNORECASE)
    if match:
        city = match.group(1).strip(' ,;')
        if city:
            return city
    return 'Cali, Colombia'

def is_ollama_connection_error(exc: Exception) -> bool:
    text = str(exc).lower()
    signals = [
        'all connection attempts failed',
        'connection refused',
        'failed to connect',
        'connecterror',
        'could not connect',
    ]
    return any(signal in text for signal in signals)

def extract_text_content(content) -> str:
    if content is None:
        return ''
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = [extract_text_content(item) for item in content]
        return '\n'.join(part for part in parts if part)
    if isinstance(content, tuple):
        parts = [extract_text_content(item) for item in content]
        return '\n'.join(part for part in parts if part)
    if isinstance(content, dict):
        if 'text' in content and content['text'] is not None:
            return extract_text_content(content['text'])
        if 'content' in content and content['content'] is not None:
            return extract_text_content(content['content'])
        if 'result' in content and content['result'] is not None:
            return extract_text_content(content['result'])
        return str(content)
    text_attr = getattr(content, 'text', None)
    if text_attr is not None:
        return extract_text_content(text_attr)
    content_attr = getattr(content, 'content', None)
    if content_attr is not None:
        return extract_text_content(content_attr)
    result_attr = getattr(content, 'result', None)
    if result_attr is not None:
        return extract_text_content(result_attr)
    return str(content)

def normalize_tool_output(tool_output) -> str:
    return extract_text_content(tool_output)

def fallback_tool_answer(tool_name: str, tool_output, llm_error: str | None = None) -> str:
    text = normalize_tool_output(tool_output)
    if llm_error:
        return (
            f"Resultado directo de la herramienta {tool_name}:\n{text}\n\n"
            f"Nota: no pude contactar Ollama para redactar una respuesta ({llm_error})."
        )
    return f"Resultado directo de la herramienta {tool_name}:\n{text}"

async def run_selected_tool(tool_name: str, question: str):
    tool = tool_from_name(tool_name)
    if tool is None:
        raise ValueError(f'Herramienta no encontrada: {tool_name}')

    if tool_name == 'calculadora':
        expression = infer_calculator_expression(question)
        if not expression:
            raise ValueError(
                'No pude inferir una expresion aritmetica del mensaje. '
                'Escribe algo como: "Cuanto es (125 * 17) + 938?".'
            )
        payload = {'expression': expression}
    elif tool_name == 'clima_actual':
        payload = {'city': infer_city(question)}
    else:
        raise ValueError(f'Herramienta no soportada: {tool_name}')

    return await tool.ainvoke(payload)

def tool_message_used(messages) -> bool:
    return any(getattr(msg, 'type', '') == 'tool' for msg in messages)

async def synthesize_from_tool(question: str, tool_name: str, tool_output):
    try:
        normalized_tool_output = normalize_tool_output(tool_output)
        synthesis = llm.invoke([
            ('system', 'Responde en espanol, de forma breve y usando solamente el resultado de la herramienta.'),
            ('user', f"Pregunta: {question}\n\nSalida de {tool_name}: {normalized_tool_output}\n\nRedacta una respuesta final clara."),
        ])
        return extract_text_content(synthesis.content)
    except Exception as exc:
        if is_ollama_connection_error(exc):
            return fallback_tool_answer(tool_name, tool_output, llm_error=str(exc))
        raise

Global Popen patch skipped outside Colab


## Del agente a la interfaz conversacional

Aquí reaparece una idea de la sesión de RAG: Gradio maneja historial, mientras reconstruimos mensajes para el agente.

En esta versión añadimos controles de estrategia para clase:

- Autónomo: el agente decide libremente.
- Guiado: sugerimos tool preferida y permitimos fallback opcional.
- Determinístico: ejecutamos una tool explícita y luego redactamos la respuesta final.

Nota de diseño de UX: el único input principal es el chat. No pedimos campos separados para cálculo o clima porque la intención idealmente se infiere desde el mensaje del usuario. El selector de tool funciona como preferencia/guía, no como un formulario obligatorio.

In [18]:
import gradio as gr

def history_to_messages(question, chat_history):
    messages = []
    for item in (chat_history or []):
        role = item.get('role')
        content = item.get('content')
        if role in {'user', 'assistant'} and content is not None:
            messages.append((role, normalize_content(content)))
    messages.append(('user', question))
    return messages

def normalize_content(content):
    return extract_text_content(content)

def compact_trace_text(trace: dict):
    lines = [
        f"modo={trace.get('mode')}",
        f"used_tool={trace.get('used_tool')}",
        f"tool_selected={trace.get('tool_selected')}",
        f"guided_fallback_used={trace.get('guided_fallback_used')}",
    ]
    if trace.get('tool_error'):
        lines.append(f"tool_error={trace.get('tool_error')}")
    return ' | '.join(lines)

async def recover_with_direct_tool_if_possible(question: str, selected_tool: str | None, trace: dict, llm_exc: Exception):
    if selected_tool is None:
        return None
    try:
        tool_output = await run_selected_tool(selected_tool, question)
        trace['used_tool'] = True
        trace['guided_fallback_used'] = True
        return fallback_tool_answer(selected_tool, tool_output, llm_error=str(llm_exc))
    except Exception:
        return None

async def respond(
    question,
    chat_history,
    mode,
    preferred_tool,
    guided_fallback,
    show_trace,
 ):
    chat_history = chat_history or []
    mode = mode or 'Guided'
    messages = history_to_messages(question, chat_history)
    selected_tool = select_tool_name(question, preferred_tool)
    trace = {
        'mode': mode,
        'tool_selected': selected_tool,
        'used_tool': False,
        'guided_fallback_used': False,
        'tool_error': None,
    }

    try:
        if mode == 'Deterministic':
            if selected_tool is None:
                answer = (
                    'No pude inferir una herramienta automáticamente. '
                    'Selecciona una herramienta o ajusta tu pregunta.'
                )
            else:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True

        elif mode == 'Guided':
            guided_messages = messages
            if selected_tool is not None:
                guided_messages = [
                    (
                        'system',
                        f"Si la pregunta requiere cálculo o datos externos, prioriza usar la herramienta {selected_tool} antes de responder.",
                    )
                ] + guided_messages

            result = await agent_mcp.ainvoke({'messages': guided_messages})
            used_tool = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)
            trace['used_tool'] = used_tool

            if guided_fallback and (not used_tool) and selected_tool is not None:
                tool_output = await run_selected_tool(selected_tool, question)
                answer = await synthesize_from_tool(question, selected_tool, tool_output)
                trace['used_tool'] = True
                trace['guided_fallback_used'] = True

        else:
            result = await agent_mcp.ainvoke({'messages': messages})
            trace['used_tool'] = tool_message_used(result['messages'])
            answer = normalize_content(result['messages'][-1].content)

    except Exception as exc:
        trace['tool_error'] = str(exc)
        if is_ollama_connection_error(exc):
            recovered = await recover_with_direct_tool_if_possible(question, selected_tool, trace, exc)
            if recovered is not None:
                answer = recovered
            else:
                answer = (
                    'No pude conectar con Ollama para generar respuesta del agente. '
                    'Inicia Ollama con `ollama serve` o haz una pregunta que active calculadora/clima.'
                )
        else:
            answer = f"Ocurrió un error: {exc}"

    answer = normalize_content(answer)

    if show_trace:
        answer = f"{answer}\n\n---\nTraza MCP: {compact_trace_text(trace)}"

    updated_history = chat_history + [
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]
    return '', updated_history

def reset_chat():
    return '', []

## Lanzando la interfaz de chat

Esta versión es opcional precisamente porque ya no enseña un concepto nuevo de NLP o de MCP; enseña cómo empaquetar el agente en una demo usable. Es útil para estudiantes que quieran presentar el flujo completo de extremo a extremo.

In [19]:
with gr.Blocks() as gr_blocks:
    gr.Markdown('## Chat con herramientas MCP')

    mode = gr.Dropdown(
        choices=['Guided', 'Autonomous', 'Deterministic'],
        value='Guided',
        label='Modo de ejecución',
    )
    preferred_tool = gr.Dropdown(
        choices=['Auto', 'calculadora', 'clima_actual'],
        value='Auto',
        label='Tool preferida',
    )
    guided_fallback = gr.Checkbox(
        label='En modo Guided, usar fallback de tool si el agente no la invoca',
        value=True,
    )
    show_trace = gr.Checkbox(
        label='Mostrar traza de uso MCP',
        value=False,
    )

    chatbot = gr.Chatbot(label='Historial')
    msg = gr.Textbox(
        label='¿Qué quieres preguntar?',
        placeholder='Ejemplo: ¿Cuál es la temperatura actual en Cali? o ¿Cuánto es (125 * 17) + 938?',
    )
    clear = gr.Button('Limpiar')

    msg.submit(
        respond,
        [
            msg,
            chatbot,
            mode,
            preferred_tool,
            guided_fallback,
            show_trace,
        ],
        [msg, chatbot],
    )
    clear.click(reset_chat, None, [msg, chatbot], queue=False)

gr_blocks.launch(inline=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
gr_blocks.close()


Closing server running on port: 7860


## Conclusiones

- Este notebook solo agrega una capa de presentación; el corazón sigue siendo el mismo agente con herramientas MCP.
- Gradio permite convertir el experimento técnico en una demo conversacional muy rápidamente.
- Como material opcional, ayuda a cerrar la sesión con una visión más aplicada sin sobrecargar el flujo conceptual principal.
